# CodeShield — ML Vulnerability Detection and Classification

**Goal:** classify Python source-code snippets into the six classes present in the CodeShield dataset:

- SAFE
- SQL Injection
- Command Injection
- Hardcoded Credentials
- Insecure File Handling
- Cross-Site Scripting

The model uses **source code only** as its input. `id`, `cwe`, and `source` are excluded because `cwe` is target-derived and `source` will not be available when a developer submits arbitrary code.

Pipeline:

`Source Code → Word TF-IDF (1–2 grams) → Class-balanced Linear SVM → Sigmoid Calibration → Vulnerability + Confidence → Severity/Risk Mapping`


In [ ]:
# If needed:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 1. Load the datasets

Expected repository layout when this notebook is placed under `code/backend/ml/`:

```text
code/backend/
├── data/processed/
│   ├── codeshield_dataset.csv
│   ├── train.csv
│   ├── validation.csv
│   └── test.csv
└── ml/
    └── CodeShield_ML_Training.ipynb
```

The path helper below also works if the notebook is opened from another directory by changing `DATA_DIR`.


In [ ]:
from pathlib import Path

# Change this only if your notebook is stored somewhere else.
DATA_DIR = Path("../data/processed")

full = pd.read_csv(DATA_DIR / "codeshield_dataset.csv")
train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Full:", full.shape)
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

display(train.head(2))


## 2. Dataset inspection and data quality

In [ ]:
for name, df in {"full": full, "train": train, "validation": validation, "test": test}.items():
    print(f"\n{name.upper()}: {df.shape}")
    print("Columns:", df.columns.tolist())
    display(df.isna().sum().to_frame("missing"))
    print("Duplicate rows:", df.duplicated().sum())
    print("Duplicate code snippets:", df["code"].duplicated().sum())

print("\nClass distribution:")
display(full["label"].value_counts().to_frame("count"))

print("\nCWE distribution:")
display(full["cwe"].value_counts().to_frame("count"))

print("\nSource distribution:")
display(full["source"].value_counts().to_frame("count"))


In [ ]:
# Verify the supplied splits are disjoint by ID and by exact source code.
for a_name, a, b_name, b in [
    ("train", train, "validation", validation),
    ("train", train, "test", test),
    ("validation", validation, "test", test),
]:
    print(
        f"{a_name} vs {b_name}: "
        f"ID overlap={len(set(a.id) & set(b.id))}, "
        f"code overlap={len(set(a.code) & set(b.code))}"
    )

assert len(set(train.id) & set(validation.id)) == 0
assert len(set(train.id) & set(test.id)) == 0
assert len(set(validation.id) & set(test.id)) == 0
assert len(set(train.code) & set(validation.code)) == 0
assert len(set(train.code) & set(test.code)) == 0
assert len(set(validation.code) & set(test.code)) == 0


## 3. Exploratory data analysis

In [ ]:
full["code_length"] = full["code"].astype(str).str.len()

fig, ax = plt.subplots(figsize=(9, 5))
full["label"].value_counts().sort_values().plot(kind="barh", ax=ax)
ax.set_title("CodeShield vulnerability class distribution")
ax.set_xlabel("Number of samples")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
full.groupby("label")["code_length"].median().sort_values().plot(kind="barh", ax=ax)
ax.set_title("Median source-code length by class")
ax.set_xlabel("Characters")
plt.tight_layout()
plt.show()

display(full.groupby("label")["code_length"].agg(["count", "mean", "median", "min", "max"]).round(1))


## 4. Define the ML task

This is a **six-class multiclass classification problem**.

The model receives only the source code and predicts:

- `SAFE`
- `SQL Injection`
- `Command Injection`
- `Hardcoded Credentials`
- `Insecure File Handling`
- `Cross-Site Scripting`

We deliberately do not use `cwe` as an input because it directly identifies the vulnerability family and would leak the target. `source` is also excluded because it identifies the dataset provenance rather than being part of a real developer submission.


In [ ]:
X_train = train["code"].fillna("").astype(str)
y_train = train["label"].astype(str)

X_val = validation["code"].fillna("").astype(str)
y_val = validation["label"].astype(str)

X_test = test["code"].fillna("").astype(str)
y_test = test["label"].astype(str)

def make_pipeline(C=2.0):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            sublinear_tf=True,
            max_features=50000,
            token_pattern=r"(?u)\b\w[\w$]*\b"
        )),
        ("classifier", LinearSVC(
            class_weight="balanced",
            C=C
        ))
    ])


## 5. Baselines and model selection

A linear SVM is a strong, lightweight baseline for small text/code datasets. We use:

- word-level TF-IDF
- unigrams + bigrams
- sublinear TF scaling
- class-balanced Linear SVM

The validation set is used for selecting `C`. The test set is not used here.


In [ ]:
results = []

for C in [0.5, 1.0, 1.5, 2.0, 2.5]:
    model = make_pipeline(C)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    results.append({
        "C": C,
        "Accuracy": accuracy_score(y_val, pred),
        "Macro F1": f1_score(y_val, pred, average="macro"),
        "Weighted F1": f1_score(y_val, pred, average="weighted")
    })

comparison = pd.DataFrame(results).sort_values(
    ["Macro F1", "Accuracy"], ascending=False
)
display(comparison)

best_C = float(comparison.iloc[0]["C"])
print("Selected C:", best_C)


In [ ]:
# Validation report for the selected model
validation_model = make_pipeline(best_C)
validation_model.fit(X_train, y_train)
val_pred = validation_model.predict(X_val)

print(classification_report(y_val, val_pred, digits=4))

cm_val = confusion_matrix(y_val, val_pred, labels=validation_model.classes_)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_val, annot=True, fmt="d", cmap="Blues",
    xticklabels=validation_model.classes_,
    yticklabels=validation_model.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Validation confusion matrix")
plt.tight_layout()
plt.show()


## 6. Final calibrated model

After selecting the model using validation data, we combine **train + validation** and fit the final model.

A Linear SVM does not naturally output probabilities, so we wrap it in `CalibratedClassifierCV` using sigmoid calibration. This gives CodeShield a usable confidence score.

The test set remains untouched until this point.


In [ ]:
combined = pd.concat([train, validation], ignore_index=True)

X_combined = combined["code"].fillna("").astype(str)
y_combined = combined["label"].astype(str)

final_model = CalibratedClassifierCV(
    make_pipeline(best_C),
    method="sigmoid",
    cv=5,
    n_jobs=-1
)

final_model.fit(X_combined, y_combined)

test_pred = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)
test_confidence = test_proba.max(axis=1)

print("Test accuracy:", accuracy_score(y_test, test_pred))
print("Test macro F1:", f1_score(y_test, test_pred, average="macro"))
print("Test weighted F1:", f1_score(y_test, test_pred, average="weighted"))

print("\nFinal test classification report:")
print(classification_report(y_test, test_pred, digits=4))


In [ ]:
classes = list(final_model.classes_)

cm = confusion_matrix(y_test, test_pred, labels=classes)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=classes, yticklabels=classes
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final test confusion matrix")
plt.tight_layout()
plt.show()

metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "macro_precision": precision_score(y_test, test_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_test, test_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(y_test, test_pred, average="macro"),
    "weighted_f1": f1_score(y_test, test_pred, average="weighted"),
}
display(pd.DataFrame([metrics]))


## 7. Error analysis

In [ ]:
errors = test.copy()
errors["predicted_label"] = test_pred
errors["confidence"] = test_confidence
errors = errors[errors["label"] != errors["predicted_label"]]

print("Misclassified test samples:", len(errors))
display(
    errors[["id", "label", "predicted_label", "confidence", "cwe", "source"]]
    .sort_values("confidence", ascending=False)
)

for _, row in errors.head(8).iterrows():
    print("\n" + "="*80)
    print("ID:", row["id"])
    print("Actual:", row["label"])
    print("Predicted:", row["predicted_label"])
    print("Confidence:", round(row["confidence"], 4))
    print("Code:")
    print(row["code"][:1500])


## 8. Severity / risk assessment

The supplied dataset contains vulnerability labels and CWE values, but no independent severity target.

Therefore, severity is **not trained as an ML prediction**. CodeShield uses a transparent, editable risk mapping after classification.

This can later be replaced by a learned severity model if severity-labelled training data becomes available.


In [ ]:
SEVERITY_MAP = {
    "SAFE": "NONE",
    "Command Injection": "CRITICAL",
    "SQL Injection": "HIGH",
    "Hardcoded Credentials": "HIGH",
    "Insecure File Handling": "HIGH",
    "Cross-Site Scripting": "MEDIUM",
}

CWE_MAP = {
    "SAFE": None,
    "Command Injection": "CWE-78",
    "SQL Injection": "CWE-89",
    "Hardcoded Credentials": "CWE-798",
    "Insecure File Handling": "CWE-22",
    "Cross-Site Scripting": "CWE-79",
}

def predict_code(code):
    if not isinstance(code, str) or not code.strip():
        raise ValueError("code must be a non-empty string")

    label = str(final_model.predict([code])[0])
    probs = final_model.predict_proba([code])[0]
    confidence = float(probs.max())

    return {
        "is_vulnerable": label != "SAFE",
        "vulnerability_type": label,
        "confidence": round(confidence, 4),
        "severity": SEVERITY_MAP[label],
        "cwe": CWE_MAP[label],
    }

# Example
example = '''
import sqlite3
username = request.args.get("username")
cursor.execute("SELECT * FROM users WHERE username='" + username + "'")
'''
predict_code(example)


## 9. Save model artifacts

The saved artifact contains the complete preprocessing + classifier + calibration pipeline, so inference does not require retraining.


In [ ]:
MODEL_DIR = Path("../codeshield_ml/models")
MODEL_DIR.mkdir(exist_ok=True)

artifact = {
    "model": final_model,
    "classes": classes,
    "best_C": best_C,
    "text_column": "code",
    "excluded_columns": ["id", "cwe", "source"],
    "severity_map": SEVERITY_MAP,
    "cwe_map": CWE_MAP,
}

joblib.dump(artifact, MODEL_DIR / "codeshield_vulnerability_model.joblib")

with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_type": "Calibrated Linear SVM",
        "features": "word TF-IDF unigrams + bigrams",
        "best_C": best_C,
        "classes": classes,
        "test_metrics": metrics
    }, f, indent=2)

print("Saved:", MODEL_DIR / "codeshield_vulnerability_model.joblib")
print("Saved:", MODEL_DIR / "metadata.json")


## 10. Reload and test inference

This verifies that the serialized artifact can be loaded in a fresh Python process and used without retraining.


In [ ]:
loaded = joblib.load(MODEL_DIR / "codeshield_vulnerability_model.joblib")

def predict_loaded(code):
    label = str(loaded["model"].predict([code])[0])
    probs = loaded["model"].predict_proba([code])[0]
    return {
        "is_vulnerable": label != "SAFE",
        "vulnerability_type": label,
        "confidence": round(float(probs.max()), 4),
        "severity": loaded["severity_map"][label],
        "cwe": loaded["cwe_map"][label],
    }

predict_loaded(example)


## 11. FastAPI integration

The backend can load the artifact once at application startup and reuse it for requests.

Example:

```python
from codeshield_ml.model import CodeShieldModel

model = CodeShieldModel.load("codeshield_ml/models")

result = model.predict(source_code)
```

The result can then be combined with findings from `static_rules.py` and line/file information from the scanner.


## 12. Limitations and future work

- The dataset is small (399 labelled snippets).
- The model supports the vulnerability categories represented in the dataset only.
- Confidence is calibrated probability, not proof that a vulnerability exists.
- Severity is currently a transparent rule-based mapping because the dataset has no independent severity label.
- Source-code context outside the submitted snippet can matter.
- Future versions can compare this baseline with code-aware transformer models, AST/tree-sitter features, and a stronger hybrid static-analysis + ML architecture.
